# SYDE 556/750 &mdash; Practice Notebook 3
## Nengo and Simple Dynamics

**Prepares you for: Test 3.**

This is the **first notebook that uses Nengo** (install it from <https://www.nengo.ai/getting-started/>). Up to now you have built everything from scratch. Here you will first generate a from-scratch network, then learn Nengo basics, and reproduce that network in Nengo to see how much it simplifies the code. Then you will use Nengo to explore recurrent dynamics.

**How to use it**
- **No solutions are released.** Each part ends with an *Expected result* checkpoint so you can tell whether you are on track.
- Cells marked &#x270D; are for written answers.

**Contents**
1. A vector computation, by hand
2. Building an ensemble of neurons
3. Connecting neurons (and reproducing &sect;1 in Nengo)
4. Dynamics: the neural integrator


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nengo


# 1. A vector computation, by hand

In PN2 you computed scalar functions across populations. Here you compute a 2-D **vector** function,
$$w = x - 3y + 2z - 2q,$$
using your own code (no Nengo yet). This needs five populations ($x$, $y$, $z$, $q$, $w$), each representing a 2-D value, with encoders drawn randomly around the unit circle. The decoded outputs of $x$, $y$, $z$, $q$ all feed into $w$.

&#x1F4CC; You will rebuild this *exact* network in Nengo in &sect;3 &mdash; once you've learned to make ensembles (&sect;2) and connect them (&sect;3) &mdash; and then compare the two.


**a) Constant inputs.** Using your population and transformation code from PN2 (extended to 2-D), compute $w = x - 3y + 2z - 2q$ for
$$x = (0.5, 1),\quad y = (0.1, 0.3),\quad z = (0.2, 0.1),\quad q = (0.4, -0.2).$$
Plot the decoded $\hat{w}$ and the ideal $w$ over 1 s (plot both components).


In [ ]:
# Reuse your PN2 population and transformation machinery, extended to 2-D vectors.
# Five 2-D populations; decode the linear combination into w; constant inputs.
# TODO


**b) Sinusoidal inputs.** Repeat (a) for the time-varying inputs
$$x = (0.5, 1),\quad y = (\sin(4\pi t), 0.3),\quad z = (0.2, 0.1),\quad q = (\sin(4\pi t), -0.2).$$


In [ ]:
# Same by-hand network with the time-varying inputs above.
# TODO


# 2. Building an ensemble of neurons

Make a single Nengo ensemble: **100 neurons**, **1-D**, intercepts in $[-1, 1]$, maximum rates in $[100, 200]$ Hz, $\tau_{RC} = 20$ ms, $\tau_{\mathrm{ref}} = 2$ ms. (You don't need to run this model over time.)


**a) Tuning curves & accuracy.** Plot the population tuning curves and the representation-accuracy plot ($x - \hat{x}$); report the RMSE.

&#x1F40D; See the Nengo "tuning curves" tutorial at <https://www.nengo.ai/nengo/v4.0.0/examples/usage/tuning-curves.html>. A quick way to get $\hat{x}$: make a dummy identity connection from the ensemble to itself so Nengo computes the population decoders (under noise $\sigma = 0.1\cdot 200$ Hz), then use `nengo.utils.connection.eval_point_decoding`.


In [ ]:
# Build the ensemble; plot tuning curves; compute x_hat and plot (x - x_hat); report RMSE.
# TODO


**b) RMSE vs. radius.** Compute the RMSE for radii 0.5, 1, 2, 4 and plot it. Keep the same (relative) intercepts and max rates across experiments.

&#x1F4CC; Fix the network seed so the neuron parameters stay the same across runs; Nengo rescales intercepts with the radius automatically.


In [ ]:
# RMSE for radius in [0.5, 1, 2, 4]; plot RMSE vs radius.
# TODO


**c) Discussion.** What mathematical relationship between radius and RMSE do you observe (write an equation)? Why?

&#x270D; *Your answer here.*


**d) RMSE vs. refractory period.** How do the RMSE and tuning curves change as $\tau_{\mathrm{ref}}$ varies over 1&ndash;5 ms? Plot tuning curves for at least four values and an RMSE-vs-$\tau_{\mathrm{ref}}$ curve (same ensemble parameters throughout).


In [ ]:
# Vary tau_ref in [1, 5] ms; plot tuning curves (>=4 values) and RMSE vs tau_ref.
# TODO


**e) RMSE vs. membrane time constant.** Same as (d) but varying $\tau_{RC}$ over 10&ndash;100 ms.

In [ ]:
# Vary tau_RC in [10, 100] ms; plot tuning curves (>=4 values) and RMSE vs tau_RC.
# TODO


**f) Discussion.** Describe what happens to the tuning curves as $\tau_{\mathrm{ref}}$ and $\tau_{RC}$ change, and explain why those shape changes affect the RMSE the way you observe.

&#x270D; *Your answer here.*


# 3. Connecting neurons

Make a second ensemble of spiking neurons with the same parameters as in &sect;2 but only **50 neurons**. Connect the first ensemble to the second, using a post-synaptic time constant of 10 ms on the connection and a 10 ms synapse on your probes. Drive the first ensemble with a step input that is 1 for $0.1 < t < 0.4$ s and 0 otherwise.

&#x1F40D; You must use a `nengo.Node` to feed an arbitrary function into the network. You can use a Python `lambda` to write this very concisely. For example,
```python
nd_input = nengo.Node(lambda t: 0.0 if t < 0.5 else (1.0 if t < 1.0 else 0.0))
```
creates a step function that is 1 for $0.5 < t < 1.0$ and 0 otherwise. Instead of a lambda, you can also use the `Piecewise` process built into Nengo:
```python
nd_input = nengo.Node(nengo.processes.Piecewise({
    0.5: 1.0,
    1.0: 0.0,
}))
```


**a) Identity function.** Connect the ensembles so the second computes the identity of the first. Show the input value and the decoded values of both ensembles in three separate plots. Run for 0.5 s.


In [ ]:
# Two ensembles (100 -> 50), identity connection, tau=10 ms; step input; 3 plots; run 0.5 s.
# TODO


**b) Affine transformation.** Make a version where the second ensemble computes $y(t) = 1 - 2x(t)$ instead. Show the same three plots.

In [ ]:
# Same network, connection computing y = 1 - 2x.
# TODO


**c) Reproduce your &sect;1 network in Nengo.** You now have the pieces to rebuild the by-hand network from &sect;1. Make five 2-D ensembles ($x$, $y$, $z$, $q$, $w$) and connect $x$, $y$, $z$, $q$ into $w$ with the transforms that compute $w = x - 3y + 2z - 2q$. Run it for the constant inputs from &sect;1a and then the sinusoidal inputs from &sect;1b, and overlay the Nengo result on your by-hand result.


In [ ]:
model = nengo.Network(seed=556)
with model:
    # Five 2-D ensembles (e.g., 200 neurons each): x, y, z, q, w
    # Input Nodes supplying the four input vectors (constant first, then sinusoidal)
    # Connect each input ensemble into w with the appropriate transform
    #   (the coefficients of w = x - 3y + 2z - 2q)
    # Probe w's decoded output (synapse ~0.01)
    # TODO
    pass

with nengo.Simulator(model) as sim:
    sim.run(1.0)

# Overlay the Nengo-decoded w on your by-hand result from Section 1.
# TODO


**d) Discussion.** How much less code was the Nengo version? Do the two results agree (and where do they differ)? In your own words, what is Nengo doing under the hood that you did by hand in &sect;1?

&#x270D; *Your answer here.*

*Expected:* the two networks should agree closely. This is your validation that Nengo implements exactly the representation/transformation math you built by hand. From here on we lean on Nengo.


# 4. Dynamics: the neural integrator

Build a neural integrator: one ensemble (**200 neurons**, parameters as in &sect;2), an input, a connection from the input to the ensemble, and a recurrent connection from the ensemble back to itself. To be an integrator, the desired dynamical system is $\frac{dx(t)}{dt} = u$, where $x$ is the state and $u$ is the input.

The recurrent post-synaptic time constant is $\tau = 50$ ms, and the input post-synaptic time constant is 5 ms. Having two different time constants violates what we assumed when deriving Principle 3; use the longer time constant $\tau$ (i.e., $\tau$ from the recurrent connection) when computing $A'$ and $B'$. Part of this question is to explore how having two different time constants affects the quality of the result compared to what we saw in lecture.

&#x1F4CC; For all probes, use a synapse of 10 ms.
&#x1F4CC; To answer the questions, you may want to run the experiments a few times with different random numbers. The behaviour of the system may be different in each run. Think about what this means regarding the sources of error you observe.


**a) Transform the dynamical system.** Rewrite the integrator as $\frac{d\vec{x}}{dt} = A\vec{x} + B\vec{u}$, i.e., give the matrices $A$ and $B$ (you may just use the equations from class; you do not have to re-derive them). What are the matrices $A'$ and $B'$ you must use to implement the system through the recurrent post-synaptic filter?

&#x1F4CC; You may assume $\vec{u}$ and $\vec{x}$ are $d$-dimensional and write the equations as generally as possible. For the rest of this question we take $d = 1$.

&#x270D; *Your derivation here.* (You will use $A'$ and $B'$ in the code below.)


**b) Integrator (spiking).** Drive the integrator with an input of 0.9 from $t = 0.04$ to $t = 1.0$ s (0 otherwise) and run for 1.5 s. Plot the input, the ideal integral, and the value represented by the ensemble.

&#x1F40D; We will run this simulation with various input functions and ensemble parameters in the following parts. It is highly recommended that you write one function that takes the ensemble parameters and the input function as arguments and produces all the plots you need.


In [ ]:
tau = 0.05      # recurrent synapse
tau_in = 0.005  # input synapse

model = nengo.Network(seed=556)
with model:
    # stim Node (the 0.9 step described above)
    # x = nengo.Ensemble(200, 1)
    # input connection (synapse=tau_in) with transform B'
    # recurrent connection x -> x (synapse=tau) with transform A'
    # probe x with a 10 ms synapse
    # TODO  (use your A', B' from part a)
    pass

# Run 1.5 s; plot input, ideal integral, and the represented value.
# TODO


**c) Discussion.** What is the expected ideal result, i.e., if you just mathematically computed the integral of the input, what equation describes it? How does the simulated output compare to that ideal?

&#x1F4CC; When you write down the integral of the step function, you have to distinguish between three cases. Your function should be a *definite* integral of the form
$$\int_{0}^{t} u(t')\,dt' = \begin{cases} \ldots & \text{if } t < 0.04 \\ \ldots & \text{if } 0.04 \le t < 1 \\ \ldots & \text{if } 1 \le t. \end{cases}$$
&#x1F4CC; Make sure you understand the difference between a *definite* and an *indefinite* integral before continuing.
&#x1F4CC; Plot the ideal on top of your simulation result from the previous part.
&#x1F40D; You can use `np.cumsum` to compute the ideal integral numerically (don't forget to scale by `dt`). However, still write down the integral mathematically.

&#x270D; *Your answer here.*


**d) Rate mode.** Re-run (b) in rate mode (pass `neuron_type=nengo.LIFRate()` to the ensemble). Show the same plots.

In [ ]:
# Re-run the integrator in rate mode.
# TODO


**e) Discussion.** How does the rate-mode result compare to (b)? What deviations from the ideal remain, and where do they come from?

&#x270D; *Your answer here.*


**f) Shorter input pulse.** Back in spiking mode, change the input to 0.9 from $t = 0.04$ to $0.16$ s. Show the same plots.

In [ ]:
# Integrator with a short input pulse.
# TODO


**g) Discussion.** How does this compare to (b)? What is the ideal equation? Does it behave as intended? Is it better or worse?

&#x270D; *Your answer here.*


**h) Input ramp.** Change the input to a ramp from 0 to 0.9 over $t = 0$ to $0.45$ s (0 afterward). Show the same plots.

In [ ]:
# Integrator with a ramp input.
# TODO


**i) Discussion.** What does the ensemble end up representing, and why? Give the ideal equation for the curve it traces.

&#x1F4CC; As above, write down the *definite* integral $\int_0^t u(t')\,dt'$. This means there will be no integration constant in your result.

&#x270D; *Your answer here.*


**j) Sinusoidal input.** Change the input to $5\sin(5t)$. Show the same plots.

In [ ]:
# Integrator with a sinusoidal input.
# TODO


**k) Discussion.** What should the represented value be (write the equation)? How well does it do, and why does it differ from the ideal?

&#x1F4CC; Again, write down the *definite* integral $\int_0^t u(t')\,dt'$. There is no integration constant.

&#x270D; *Your answer here.*


**l) Controlled integrator.** Implement a controlled integrator over two populations (the feedback gain is itself an input). Demonstrate that it works as expected.

In [ ]:
# Controlled integrator.
# TODO


**(Optional) m) Nonlinear dynamics.** Implement a nonlinear dynamical system we have not seen in class. Write (i) its differential equation, (ii) the transformed equations for the post-synaptic filter, (iii) the Nengo implementation (spiking), and (iv) show the simulation matches the expected behaviour at least qualitatively.

In [ ]:
# (Optional) a nonlinear dynamical system of your choosing.
# TODO


---
*End of Practice Notebook 3.* Test 3 will assume you can do everything above by hand or from scratch.
